# Proyecto práctico: árbol de decisión y random forest con scikit-learn

In [209]:
#Importamos las librerias principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Utilizaremos el **Car Evaluation Data Set** de Kaggle: https://www.kaggle.com/datasets/elikplim/car-evaluation-data-set

In [210]:
# https://archive.ics.uci.edu/dataset/19/car+evaluation

!pip install ucimlrepo

In [211]:
#Cargamos dataset a utilizar

from ucimlrepo import fetch_ucirepo

# fetch dataset
car_evaluation = fetch_ucirepo(id=19)

# data (as pandas dataframes)
X = car_evaluation.data.features.copy()
y = car_evaluation.data.targets.copy()

## Análisis exploratorio de datos

In [212]:
#Visualizacion del dataframe

print(X.head())
print(y.head())


  buying  maint doors persons lug_boot safety
0  vhigh  vhigh     2       2    small    low
1  vhigh  vhigh     2       2    small    med
2  vhigh  vhigh     2       2    small   high
3  vhigh  vhigh     2       2      med    low
4  vhigh  vhigh     2       2      med    med
   class
0  unacc
1  unacc
2  unacc
3  unacc
4  unacc


In [213]:
# variable information
car_evaluation.variables

,name,role,type,demographic,description,units,missing_values
0,buying,Feature,Categorical,None,buying price,None,no
1,maint,Feature,Categorical,None,price of the maintenance,None,no
2,doors,Feature,Categorical,None,number of doors,None,no
3,persons,Feature,Categorical,None,capacity in terms of persons to carry,None,no
4,lug_boot,Feature,Categorical,None,the size of luggage boot,None,no
5,safety,Feature,Categorical,None,estimated safety of the car,None,no
6,class,Target,Categorical,None,"evaulation level (unacceptable, acceptable, go...",None,no


El mismo creador del dataframe indica que no hay valores nulos e indica que todas las variables son categoricas.

In [214]:
X.shape

(1728, 6)

In [215]:
y.shape

(1728, 1)

Al haber mas de 50 registros no nulos, se puede usar estimador (https://scikit-learn.org/stable/machine_learning_map.html)



In [216]:
car_evaluation.variables

,name,role,type,demographic,description,units,missing_values
0,buying,Feature,Categorical,None,buying price,None,no
1,maint,Feature,Categorical,None,price of the maintenance,None,no
2,doors,Feature,Categorical,None,number of doors,None,no
3,persons,Feature,Categorical,None,capacity in terms of persons to carry,None,no
4,lug_boot,Feature,Categorical,None,the size of luggage boot,None,no
5,safety,Feature,Categorical,None,estimated safety of the car,None,no
6,class,Target,Categorical,None,"evaulation level (unacceptable, acceptable, go...",None,no


In [217]:
X.head()

,buying,maint,doors,persons,lug_boot,safety
0,vhigh,vhigh,2,2,small,low
1,vhigh,vhigh,2,2,small,med
2,vhigh,vhigh,2,2,small,high
3,vhigh,vhigh,2,2,med,low
4,vhigh,vhigh,2,2,med,med


In [218]:
X['buying'].unique()

array(['vhigh', 'high', 'med', 'low'], dtype=object)

las variables no vienen como ordinales

In [219]:
tamano = ["low", "med", "high", "vhigh"]

X["buying"] = pd.Categorical(
    X["buying"],
    categories = tamano,
    ordered = True
    )

In [220]:
X['maint'].unique()

array(['vhigh', 'high', 'med', 'low'], dtype=object)

In [221]:
X["maint"] = pd.Categorical(
    X["maint"],
    categories = tamano,
    ordered = True
    )

In [222]:
X['doors'].unique()

array(['2', '3', '4', '5more'], dtype=object)

In [223]:
X["doors"] = pd.Categorical(
    X["doors"],
    categories = ['2', '3', '4', '5more'],
    ordered = True
    )

In [224]:
X['persons'].unique()

array(['2', '4', 'more'], dtype=object)

In [225]:
X["persons"] = pd.Categorical(
    X["persons"],
    categories = ['2', '4', 'more'],
    ordered = True
    )

In [226]:
X['lug_boot'].unique()

array(['small', 'med', 'big'], dtype=object)

In [227]:
X["lug_boot"] = pd.Categorical(
    X["lug_boot"],
    categories = ['small', 'med', 'big'],
    ordered = True
    )

In [228]:
X['safety'].unique()

array(['low', 'med', 'high'], dtype=object)

In [229]:
X["safety"] = pd.Categorical(
    X["safety"],
    categories = ['low', 'med', 'high'],
    ordered = True
    )

In [230]:
y['class'].unique()

array(['unacc', 'acc', 'vgood', 'good'], dtype=object)

In [231]:
y['class'] = pd.Categorical(
    y['class'],
    categories = ['unacc', 'acc', 'vgood', 'good'],
    ordered = True
    )

Primer resumen de los datos:
* Hay 7 variables en el conjunto de datos. Todas las variables son de tipo de datos categóricos.
* Estos se dan por compra, mantenimiento, puertas, personas, lug_boot, seguridad y clase.
* La clase es la variable de destino o target.

In [232]:
X.columns

Index(['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety'], dtype='object')

In [233]:
# Exploremos un poco mas la variable target
print(X['buying'].value_counts(normalize=True))
print(X['maint'].value_counts(normalize=True))
print(X['doors'].value_counts(normalize=True))
print(X['persons'].value_counts(normalize=True))
print(X['lug_boot'].value_counts(normalize=True))
print(X['safety'].value_counts(normalize=True))

print(y['class'].value_counts(normalize=True))

buying
low      0.25
med      0.25
high     0.25
vhigh    0.25
Name: proportion, dtype: float64
maint
low      0.25
med      0.25
high     0.25
vhigh    0.25
Name: proportion, dtype: float64
doors
2        0.25
3        0.25
4        0.25
5more    0.25
Name: proportion, dtype: float64
persons
2       0.333333
4       0.333333
more    0.333333
Name: proportion, dtype: float64
lug_boot
small    0.333333
med      0.333333
big      0.333333
Name: proportion, dtype: float64
safety
low     0.333333
med     0.333333
high    0.333333
Name: proportion, dtype: float64
class
unacc    0.700231
acc      0.222222
good     0.039931
vgood    0.037616
Name: proportion, dtype: float64


los features son proporcionales lo cual es bueno

In [234]:
# calculando ratio de imbalance

ir = y.value_counts().max() / y.value_counts().min()

ir

18.615384615384617

Según ChatGPT hay desbalanceo al ser este numero mayor a 10

## Procesamiento de datos

In [235]:
#Importamos las librerias necesarias para la creacion del modelo
from sklearn.model_selection import train_test_split

#30% para test y 70% para train
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    #train_size=0.7,
    random_state = 42 #no se por qué no pone simplemente 0
    )

In [236]:
#Veamos que obtuvimos
X_train.shape, X_test.shape

((1209, 6), (519, 6))

In [237]:
y_train.shape, y_test.shape

((1209, 1), (519, 1))

In [238]:
#Veamos que tenemos. Por ejemplo, en X_train
X_train.head()

,buying,maint,doors,persons,lug_boot,safety
1178,med,med,5more,4,big,high
585,high,high,3,more,small,low
1552,low,med,3,4,med,med
1169,med,med,5more,2,big,high
1033,med,high,4,2,big,med


## Entrenamiento de modelo de clasificación con árbol de decisión

In [239]:
#Importante: todos nuestros tipos de datos son object, realizamos una transformacion

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

name_features = X.columns.to_list().copy()

ordinales = [list(X[i].unique().categories) for i in ordinal_features]

ordinal_encoder = OrdinalEncoder(categories=ordinales)

preprocessor = ColumnTransformer(
    transformers=[
        ('ord', ordinal_encoder, name_features)
    ]
)

In [240]:
# Verificamos la transformación

preprocessor.set_output(transform="pandas")

print(preprocessor.fit_transform(X))

preprocessor.set_output(transform="default")

      ord__buying  ord__maint  ord__doors  ord__persons  ord__lug_boot  \
0             3.0         3.0         0.0           0.0            0.0   
1             3.0         3.0         0.0           0.0            0.0   
2             3.0         3.0         0.0           0.0            0.0   
3             3.0         3.0         0.0           0.0            1.0   
4             3.0         3.0         0.0           0.0            1.0   
...           ...         ...         ...           ...            ...   
1723          0.0         0.0         3.0           2.0            1.0   
1724          0.0         0.0         3.0           2.0            1.0   
1725          0.0         0.0         3.0           2.0            2.0   
1726          0.0         0.0         3.0           2.0            2.0   
1727          0.0         0.0         3.0           2.0            2.0   

      ord__safety  
0             0.0  
1             1.0  
2             2.0  
3             0.0  
4          

ColumnTransformer(transformers=[('ord',
                                 OrdinalEncoder(categories=[['low', 'med',
                                                             'high', 'vhigh'],
                                                            ['low', 'med',
                                                             'high', 'vhigh'],
                                                            ['2', '3', '4',
                                                             '5more'],
                                                            ['2', '4', 'more'],
                                                            ['small', 'med',
                                                             'big'],
                                                            ['low', 'med',
                                                             'high']]),
                                 ['buying', 'maint', 'doors', 'persons',
                                  'lug_boot', 'safety'])])

In [254]:
X

,buying,maint,doors,persons,lug_boot,safety
0,vhigh,vhigh,2,2,small,low
1,vhigh,vhigh,2,2,small,med
2,vhigh,vhigh,2,2,small,high
3,vhigh,vhigh,2,2,med,low
4,vhigh,vhigh,2,2,med,med
...,...,...,...,...,...,...
1723,low,low,5more,more,med,med
1724,low,low,5more,more,med,high
1725,low,low,5more,more,big,low
1726,low,low,5more,more,big,med


Aparentemente se ve bien

In [259]:
#Importar árbol de decisión
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

#Creacion del modelo
tree = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth = 2,
                                          class_weight="balanced",
                                          random_state=0
                                          ))
])

In [260]:
#Entrenamiento
tree.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('ord',
                                                  OrdinalEncoder(categories=[['low',
                                                                              'med',
                                                                              'high',
                                                                              'vhigh'],
                                                                             ['low',
                                                                              'med',
                                                                              'high',
                                                                              'vhigh'],
                                                                             ['2',
                                                                              '3',
                                                                              '4',
                                                                              '5more'],
                                                                             ['2',
                                                                              '4',
                                                                              'more'],
                                                                             ['small',
                                                                              'med',
                                                                              'big'],
                                                                             ['low',
                                                                              'med',
                                                                              'high']]),
                                                  ['buying', 'maint', 'doors',
                                                   'persons', 'lug_boot',
                                                   'safety'])])),
                ('classifier',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=2,
                                        random_state=0))])

In [261]:
#Calculo de las predicciones en Train y Test
y_train_pred_tree = tree.predict(X_train)
y_test_pred_tree = tree.predict(X_test)

In [262]:
y_train_pred_tree

array(['vgood', 'unacc', 'vgood', ..., 'acc', 'vgood', 'vgood'],
      dtype=object)

## Evaluación de modelo de clasificación con árbol de decisión

In [245]:
#Calculo de metricas


#Calculo el accuracy en Train


#Calculo el accuracy en Test


In [246]:
#Verificamos el feature importances


## Entrenamiento de modelo de clasificación con random forest

In [247]:
#Importar random forest


In [248]:
#Calculo de las predicciones en Train y Test


## Evaluación de modelo de clasificación con random forest

In [249]:
#Calculo de metricas


#Calculo el accuracy en Train


#Calculo el accuracy en Test


#Importante: podriamos reducir el numero de estimadores para disminuir el sobreajuste del modelo.

In [250]:
# Visualizacion de las feature importantes


In [251]:
#Grafico de barras


In [252]:
# Matriz de confusion del RF


In [253]:
#RF
